# Exemplo de uso de PySpark em Colab

**Autor:** Larissa Pocceschy Martins

**Data:** 01/03/2026

### Instalação de pyspark



In [11]:
!pip install pyspark

### Imports e construção de storage no drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("teste").getOrCreate()

from pyspark.sql.functions import avg, max, min

from pyspark.sql.functions import col, when, round

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Lendo arquivo do Drive

In [10]:
df = spark.read.csv(
    "/content/drive/MyDrive/calorie_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+---------+-------+-----+-------+-------+--------+-------------+
|protein_g|carbs_g|fat_g|fiber_g|sugar_g|calories|calorie_class|
+---------+-------+-----+-------+-------+--------+-------------+
|     22.5|   65.2| 13.7|    9.9|   26.7|   479.0|       medium|
|     15.9|   20.7| 16.8|    7.6|   15.0|   316.0|          low|
|     28.3|   62.4| 14.5|   10.8|   15.3|   524.0|       medium|
|     23.3|   18.6|  7.9|   17.0|    0.2|   250.0|          low|
|     30.8|   55.1| 31.9|   12.5|    7.2|   602.0|       medium|
+---------+-------+-----+-------+-------+--------+-------------+
only showing top 5 rows


### Análise exploratória

In [4]:
df.printSchema()

root
 |-- protein_g: double (nullable = true)
 |-- carbs_g: double (nullable = true)
 |-- fat_g: double (nullable = true)
 |-- fiber_g: double (nullable = true)
 |-- sugar_g: double (nullable = true)
 |-- calories: double (nullable = true)
 |-- calorie_class: string (nullable = true)



In [5]:
df.count()

5000

In [13]:
df.filter(col("calorie_class").isNull()).count()

0

### Agregações

In [12]:
df_agg = df.groupBy("calorie_class").agg(
    avg("calories").alias("media_calorias"),
    avg("protein_g").alias("media_proteina"),
    max("calories").alias("max_calorias"),
    min("calories").alias("min_calorias")
)

df_agg.show()

+-------------+-----------------+------------------+------------+------------+
|calorie_class|   media_calorias|    media_proteina|max_calorias|min_calorias|
+-------------+-----------------+------------------+------------+------------+
|          low|331.2540064102564|21.020993589743632|       400.0|        56.0|
|         high| 742.638036809816|31.935582822085877|       975.0|       700.0|
|       medium|520.4302033992756| 26.17550849818894|       700.0|       400.0|
+-------------+-----------------+------------------+------------+------------+



In [7]:
df_transformado = (
    df
    .withColumn(
        "protein_level",
        when(col("protein_g") < 20, "low")
        .when(col("protein_g") < 30, "medium")
        .otherwise("high")
    )
    .withColumn(
        "high_sugar_flag",
        when(col("sugar_g") > 20, 1).otherwise(0)
    )
    .withColumn(
        "calories_round",
        round(col("calories"), 0)
    )
)

### Escrevendo arquivo em parquet

In [8]:
df_transformado.write \
    .mode("overwrite") \
    .partitionBy("calorie_class", "protein_level") \
    .parquet("/content/drive/MyDrive/calorie_parquet_particionado")

### Verificação de leitura dos arquivos

In [15]:
df_low = spark.read.parquet(
    "/content/drive/MyDrive/calorie_parquet_particionado/calorie_class=high"
)

df_low.show()

+---------+-------+-----+-------+-------+--------+---------------+--------------+-------------+
|protein_g|carbs_g|fat_g|fiber_g|sugar_g|calories|high_sugar_flag|calories_round|protein_level|
+---------+-------+-----+-------+-------+--------+---------------+--------------+-------------+
|     38.5|   64.7| 32.0|    7.5|   21.6|   720.0|              1|         720.0|         high|
|     35.8|   85.0| 29.3|    3.7|   23.4|   745.0|              1|         745.0|         high|
|     33.4|   63.7| 34.6|    9.2|   12.8|   704.0|              0|         704.0|         high|
|     48.2|   68.2| 35.1|    7.3|   22.1|   815.0|              1|         815.0|         high|
|     50.9|   54.0| 27.9|    8.8|   15.4|   709.0|              0|         709.0|         high|
|     37.5|   99.9| 26.2|    6.1|   34.0|   811.0|              1|         811.0|         high|
|     33.2|   80.3| 29.6|   10.2|   15.5|   725.0|              0|         725.0|         high|
|     35.0|   90.9| 24.5|    7.1|   13.5

### Alimentos altos em proteína

In [18]:
df_low.filter(col("high_sugar_flag") == 0).orderBy(col("protein_g").desc()).limit(5).show()


+---------+-------+-----+-------+-------+--------+---------------+--------------+-------------+
|protein_g|carbs_g|fat_g|fiber_g|sugar_g|calories|high_sugar_flag|calories_round|protein_level|
+---------+-------+-----+-------+-------+--------+---------------+--------------+-------------+
|     56.4|   86.1| 23.2|   10.0|    2.4|   720.0|              0|         720.0|         high|
|     50.9|   54.0| 27.9|    8.8|   15.4|   709.0|              0|         709.0|         high|
|     49.9|   79.3| 27.4|    3.9|   15.2|   780.0|              0|         780.0|         high|
|     46.7|   92.7| 18.5|    3.9|   11.6|   741.0|              0|         741.0|         high|
|     46.5|   76.1| 36.9|    7.5|    3.9|   835.0|              0|         835.0|         high|
+---------+-------+-----+-------+-------+--------+---------------+--------------+-------------+

